In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/macro_data.parquet')
# micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
# macro_df = pd.read_parquet('/content/mdndata/macro_data.parquet') # 可以根据需要切换到 Colab 的路径

# micro_cols 是微观特征列，macro_cols 是宏观特征列，后续我们会对它们分别进行不同的预处理
micro_cols=micro_df.columns.drop(['mthcaldt', 'permno', 'mthret_lead1'])
macro_cols=macro_df.columns.drop(['date','AAA', 'BAA', 'lty', 'ltr', 'tbl'])

# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='left')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
df = df[(df['mthcaldt'] <= pd.to_datetime('2023-12-31')) & (df['mthcaldt'] >= pd.to_datetime('1990-01-31'))]  # 日期在1990-01-31到2023-12-31之间

# 1.2 目标收益率设定 (绝对禁止缩尾！)
df['target_ret_final'] = df['mthret_lead1']

# 1.3 缺失值预填充 (可在此处放置你的业务填充逻辑) 
# 可以考虑先做1.4再做1.3, 但是我觉得区别不大
# 例如：使用截面均值填充微观特征缺失值
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df.groupby('mthcaldt')[feature].transform('mean'))

# 1.4 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1], 这个未来也可以考虑别的归一化的方法
print(" -> Applying Rank Normalization to Micro Features...")
micro_tensor_cols = []
for col in micro_cols:
    norm_col = f'{col}_norm'
    # 使用 pct=True 转化为 0~1，再映射到 -1~1
    df[norm_col] = df.groupby('mthcaldt')[col].transform(
        lambda x: (x.rank(pct=True) * 2) - 1
    ).fillna(0)  # 如果全截面缺失，填0(中性)
    micro_tensor_cols.append(norm_col)

# 1.5 宏观特征：时间序列 Z-score 标准化
print(" -> Applying Z-score Normalization to Macro Features...")
macro_tensor_cols = list(macro_cols)
# macro_tensor_cols = []
# for col in macro_cols:
#     norm_col = f'{col}_norm'
#     df[norm_col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)
#     df[norm_col] = df[norm_col].fillna(0)
#     macro_tensor_cols.append(norm_col)

# 1.6 清理最终的空值行并固化数据集
processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

# # (可选) 你可以在这里将 processed_df 存为 Parquet:
# processed_df.to_parquet('./data/fully_processed_model_input.parquet')
pd.concat([processed_df.iloc[:5], processed_df.iloc[-5:]])

In [ ]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        dates = np.sort(self.df[self.date_col].unique())
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (train_dates.dt.strftime('%Y-%m').iloc[0], train_dates.dt.strftime('%Y-%m').iloc[-1]),
                'val': (val_dates.dt.strftime('%Y-%m').iloc[0], val_dates.dt.strftime('%Y-%m').iloc[-1]),
                'test': (test_dates.dt.strftime('%Y-%m').iloc[0], test_dates.dt.strftime('%Y-%m').iloc[-1])
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months

# # ==========================================
# # 3. 结构化 MDN (Structured MoE-MDN)
# # ==========================================
# class StructuredMDN(nn.Module):
#     def __init__(self, macro_dim, micro_dim, hidden_dim=64, num_components=3):
#         super(StructuredMDN, self).__init__()
#         self.num_components = num_components
        
#         # 通道 A: Macro Network (门控网络) -> 预测机制权重 pi
#         self.macro_net = nn.Sequential(
#             nn.Linear(macro_dim, hidden_dim // 2),
#             nn.LayerNorm(hidden_dim // 2),
#             nn.ELU(),
#             nn.Linear(hidden_dim // 2, num_components)
#         )
        
#         # 通道 B: Micro Network (专家网络) -> 预测特质 mu 和 sigma
#         self.micro_extractor = nn.Sequential(
#             nn.Linear(micro_dim, hidden_dim),
#             nn.BatchNorm1d(hidden_dim),
#             nn.ELU(),
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.BatchNorm1d(hidden_dim // 2),
#             nn.ELU()
#         )
        
#         self.mu_head = nn.Linear(hidden_dim // 2, num_components)
#         self.sigma_head = nn.Linear(hidden_dim // 2, num_components)

#     def forward(self, x_macro, x_micro):
#         pi_logits = self.macro_net(x_macro)
#         h_micro = self.micro_extractor(x_micro)
#         mu = self.mu_head(h_micro)
#         sigma = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6 
#         return pi_logits, mu, sigma
    
# import torch.nn as nn

class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, micro_hidden_dim=128, macro_hidden_dim=64, num_components=3, dropout_rate=0.2):
        """
        升级版结构化 MDN：
        - micro_hidden_dim 提升至 128 (为未来扩展到 100+ 特征做准备)
        - macro_hidden_dim 提升至 64 (为未来扩展到 100+ 特征做准备)
        - 加入 Dropout 防止对某个单一因子过度依赖 (过拟合)
        """
        super(StructuredMDN, self).__init__()
        self.num_components = num_components
        
        # ==========================================
        # 通道 A: Macro Network (门控网络 -> 预测机制权重 pi)
        # 宏观数据通常维度低且稳定，不需要太深
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate), # 随机屏蔽一部分宏观因子，防止死记硬背
            nn.Linear(macro_hidden_dim // 2, num_components)
        )
        
        # ==========================================
        # 通道 B: Micro Network (专家网络 -> 预测 mu 和 sigma)
        # 微观数据维度高、噪音大，稍微加深一点，并强力正则化
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, micro_hidden_dim),
            nn.BatchNorm1d(micro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.BatchNorm1d(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )
        
        # 预测头 (Heads)
        self.mu_head = nn.Linear(micro_hidden_dim // 2, num_components)
        
        # Sigma 必须严格为正，使用 Softplus
        self.sigma_head = nn.Linear(micro_hidden_dim // 2, num_components)
        self.nu_head    = nn.Linear(micro_hidden_dim // 2, num_components) # 新增：预测 t 分布的自由度 nu，增强对极端值的鲁棒性

    def forward(self, x_macro, x_micro):
        pi_logits = self.macro_net(x_macro)
        h_micro = self.micro_extractor(x_micro)
        
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6 # softplus=log(1+exp(x)), 可以确保输出严格为正，避免数值不稳定
        # 在金融学中，如果 nu <= 2，分布的方差将无穷大（不存在），这会导致后续计算崩溃。
        # softplus 保证了正数，+ 2.01 是为了建立一道绝对的物理防火墙，确保方差永远收敛！
        nu        = nn.functional.softplus(self.nu_head(h_micro)) + 2.01
        
        return pi_logits, mu, sigma, nu
    
# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, target, weights, nu):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)

    # # Normal 分布的 log_prob
    # normal_dist = torch.distributions.Normal(mu, sigma)
    # log_normal = normal_dist.log_prob(target.expand_as(mu))
    # log_mix = log_pi + log_normal

    # Student's t 分布的 log_prob
    t_dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma)
    log_t_prob  = t_dist.log_prob(target.expand_as(mu))
    log_mix     = log_pi + log_t_prob
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.mean()

# ==========================================
# 5. 极速版双通道训练循环 (All-in-VRAM 加速)
# ==========================================
def train_structured_mdn(model, 
                         X_train_macro, X_train_micro, y_train, w_train, 
                         X_val_macro, X_val_micro, y_val, w_val, 
                         epochs=500, batch_size=8192, lr=1e-3, patience=20): # 注意 batch_size 变了！
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # # 🌟 极速黑客技巧：直接在放入 Dataset 前，把所有的矩阵一次性全部送到 GPU！
    # # 这样在后面的 for 循环中，完全没有 CPU 到 GPU 的传输延迟
    # tr_dataset = TensorDataset(
    #     torch.FloatTensor(X_train_macro).to(device), 
    #     torch.FloatTensor(X_train_micro).to(device), 
    #     torch.FloatTensor(y_train).unsqueeze(1).to(device),
    #     torch.FloatTensor(w_train).unsqueeze(1).to(device)
    # )
    
    # va_dataset = TensorDataset(
    #     torch.FloatTensor(X_val_macro).to(device), 
    #     torch.FloatTensor(X_val_micro).to(device), 
    #     torch.FloatTensor(y_val).unsqueeze(1).to(device),
    #     torch.FloatTensor(w_val).unsqueeze(1).to(device)
    # )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6,
    )
    # 极度清爽，直接将传入的 Tensor 打包，不需要任何格式转换和设备转移！
    tr_dataset = TensorDataset(X_train_macro, X_train_micro, y_train.unsqueeze(1), w_train.unsqueeze(1))
    va_dataset = TensorDataset(X_val_macro, X_val_micro, y_val.unsqueeze(1), w_val.unsqueeze(1))
    
    # 因为数据已经在显卡里了，不需要复杂的 worker
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    # 🌟 新增：用于记录 Loss 历史
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        # 注意看：这里去掉了 .to(device)！因为数据切片本来就在显存里了！
        for b_macro, b_micro, b_y, b_w in train_loader:
            optimizer.zero_grad()
            
            # pi_logits, mu, sigma = model(b_macro, b_micro)

            pi_logits, mu, sigma, nu = model(b_macro, b_micro)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w, nu)

            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_y, b_w in val_loader:
                pi_logits, mu, sigma, nu = model(b_macro, b_micro)
                loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w, nu)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
        val_loss_history.append(val_loss)

        scheduler.step(val_loss)
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history

# ==========================================
# 6. 启动模型训练流水线
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    参数：
    df_dates: pd.Series, 包含 mthcaldt 日期
    half_life_months: 半衰期(月)。默认 36 个月(3年)权重减半。
    """
    max_date = df_dates.max()
    # 计算每个样本距离该窗口最新日期的月数差
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    
    # 衰减因子 lambda: (0.5)^(1/half_life)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    
    # 计算基础权重
    weights = np.power(decay_factor, months_diff).values
    
    # 【极度关键】归一化权重，使得权重的均值为 1。
    # 如果不归一化，整个 Loss 的绝对值会变小，导致梯度的步长变相缩小！
    weights = weights / weights.mean() 
    return weights

In [ ]:
if __name__ == "__main__":
    seed_everything(seed=42)
    distribution_regime = "t_dist"  # 选择分布类型：'normal' 或 't_dist'
    print("\n>>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...")
        
    # 1. 实例化数据窗口生成器 (由于我们要用 Index 切片，重置一下 df 的索引非常重要)
    processed_df = processed_df.reset_index(drop=True)
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    window_gen = generator.expanding_window_generator(initial_train_years=15, val_years=1, test_years=1)

    # ==============================================================
    # 🌟 终极黑客技巧：全局数据一次性打入 GPU (Global All-in-VRAM)
    # ==============================================================
    print("    -> Pushing the entire 30-year dataset into GPU VRAM at once...")
    global_X_mac = torch.FloatTensor(processed_df[macro_tensor_cols].values).to(device)
    global_X_mic = torch.FloatTensor(processed_df[micro_tensor_cols].values).to(device)
    global_Y     = torch.FloatTensor(processed_df['target_ret_final'].values).to(device)
    # ==============================================================

    history_logs = []
    all_oos_predictions = []
    best_k = 3
    macro_dim = len(macro_tensor_cols)
    micro_dim = len(micro_tensor_cols)
    global_model = StructuredMDN(macro_dim, micro_dim, num_components=best_k).to(device)

    print("\n>>> [3/3] Starting Expanding Window Structured MDN Training (Zero PCIe Overhead)...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Val: {info['val'][0]}~{info['val'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        # --- 记录开始时间 ---
        start_time = time.time()

        # 提取各个集合在全局大表中的行索引 (Index)
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # tr_mac_raw = global_X_mac[idx_tr]
        # mac_mean   = tr_mac_raw.mean(dim=0, keepdim=True)   # shape: (1, macro_dim)
        # mac_std    = tr_mac_raw.std(dim=0, keepdim=True) + 1e-8
 
        # X_tr_mac = (global_X_mac[idx_tr] - mac_mean) / mac_std
        # X_va_mac = (global_X_mac[idx_va] - mac_mean) / mac_std
        # X_te_mac = (global_X_mac[idx_te] - mac_mean) / mac_std


        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, Y_tr = global_X_mic[idx_tr], global_Y[idx_tr]
        X_va_mic, Y_va = global_X_mic[idx_va], global_Y[idx_va]
        X_te_mic       = global_X_mic[idx_te]

        # # --- 显存内部极速切片 (零搬运延迟) ---
        # X_tr_mac, X_tr_mic, Y_tr = global_X_mac[idx_tr], global_X_mic[idx_tr], global_Y[idx_tr]
        # X_va_mac, X_va_mic, Y_va = global_X_mac[idx_va], global_X_mic[idx_va], global_Y[idx_va]
        # X_te_mac, X_te_mic       = global_X_mac[idx_te], global_X_mic[idx_te]
        
        # --- 唯一需要传输的是这一轮的动态权重 (体积极小，几乎0耗时) ---
        w_train_np = calculate_time_decay_weights(train_df['mthcaldt'], half_life_months=36)
        w_val_np   = calculate_time_decay_weights(val_df['mthcaldt'], half_life_months=12)
        w_train = torch.FloatTensor(w_train_np).to(device)
        w_val   = torch.FloatTensor(w_val_np).to(device)
        
        # 动态学习率
        current_lr = 1e-3 if window_idx == 0 else 3e-4
        
        # 训练模型 (此时传入的全是已经在显存里的 Tensor，速度起飞)
        global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
            model=global_model,     
            X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, y_train=Y_tr, w_train=w_train,
            X_val_macro=X_va_mac, X_val_micro=X_va_mic, y_val=Y_va, w_val=w_val,
            lr=current_lr           
        )
        # --- 记录结束时间 ---
        end_time = time.time()
        duration = end_time - start_time
        # --- 记录指标 ---
        history_logs.append({
            'window': window_idx + 1,
            'train_period': f"{info['train'][0]} to {info['train'][1]}",
            'test_period': f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec': duration,
            'best_val_loss': best_v_loss,
            'epochs_run': len(tr_hist),
            'train_loss_trace': tr_hist, # 存储为一个 list，方便后续绘图
            'val_loss_trace': va_hist
        })
        
        # --- 样本外预测 ---
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma, nu = global_model(X_te_mac, X_te_mic)
            pi = torch.softmax(pi_logits, dim=-1)
            
        # --- 结果回传至 CPU 保存 ---
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['pi_logits_vec'] = [json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()]  # 新增
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        preds_df['nu_vec']        = [json.dumps(vec.tolist()) for vec in nu.cpu().numpy()]
        all_oos_predictions.append(preds_df)
        
        # 测试用，仅跑几个窗口
        # if window_idx == 3: break

    print("\n>>> Pipeline complete! Aggregating all Out-of-Sample predictions...")
    if len(all_oos_predictions) > 0:
        final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
        metrics_df = pd.DataFrame(history_logs)
        print(f"Successfully generated {len(final_oos_df)} predictions.")

In [ ]:
# 指定保存路径
kaggle_path = '/kaggle/working'
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# 执行保存
final_oos_df.to_parquet(os.path.join(kaggle_path, f'final_oos_df_{timestamp}.parquet'), index=False)
metrics_df.to_parquet(os.path.join(kaggle_path, f'metrics_df_{timestamp}.parquet'), index=False)

print(f"保存成功！文件位置: {kaggle_path}")


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# %%
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/macro_data.parquet')
micro_cols = micro_df.columns.drop(['mthcaldt', 'permno', 'mthret_lead1'])
macro_cols = macro_df.columns.drop('date')

# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='left')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
df = df[(df['mthcaldt'] <= pd.to_datetime('2023-12-31')) & (df['mthcaldt'] >= pd.to_datetime('1990-01-31'))]

# 1.2 目标收益率设定 (绝对禁止缩尾！)
df['target_ret_final'] = df['mthret_lead1']

# 1.3 缺失值预填充
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df.groupby('mthcaldt')[feature].transform('mean'))

# 1.4 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1]
print(" -> Applying Rank Normalization to Micro Features...")
micro_tensor_cols = []
for col in micro_cols:
    norm_col = f'{col}_norm'
    df[norm_col] = df.groupby('mthcaldt')[col].transform(
        lambda x: (x.rank(pct=True) * 2) - 1
    ).fillna(0)
    micro_tensor_cols.append(norm_col)

# ==========================================
# [修复 Bug 2] 1.5 宏观特征：不在此处做全局 Z-score 归一化！
# 原因：用全样本（含未来）的均值/标准差会造成数据泄露，
#       导致回测结果虚高。改为在训练循环内，仅用每个窗口的
#       训练期数据来计算均值/标准差，再分别归一化 train/val/test。
# ==========================================
print(" -> Macro Features: skipping global normalization to prevent data leakage.")
print("    (Will be normalized per-window using training data only, inside the loop.)")

# 宏观特征仅做缺失值处理（前向填充，无未来信息）
macro_tensor_cols = list(macro_cols)
for col in macro_cols:
    # 先在时间上排好序后做前向填充，最后用 0 兜底（均值中性）
    df[col] = df.sort_values('mthcaldt').groupby('mthcaldt')[col].transform('first')
    df[col] = df[col].ffill().fillna(0)

# 1.6 清理最终的空值行并固化数据集
processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

pd.concat([processed_df.iloc[:5], processed_df.iloc[-5:]])


# %%
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col

    def expanding_window_generator(self, initial_train_years=15, val_years=1, test_years=1):
        dates = np.sort(self.df[self.date_col].unique())
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12

        start_idx = 0
        current_split_idx = initial_train_months

        # current_split_idx + val_months < len(dates) 保证 val 结束后至少还有 1 个 test 月
        while current_split_idx + val_months < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates   = dates[current_split_idx : current_split_idx + val_months]
            test_dates  = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]

            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data   = self.df[self.df[self.date_col].isin(val_dates)]
            test_data  = self.df[self.df[self.date_col].isin(test_dates)]

            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'),  pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val':   (pd.Timestamp(val_dates[0]).strftime('%Y-%m'),    pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test':  (pd.Timestamp(test_dates[0]).strftime('%Y-%m'),   pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months


# ==========================================
# 3. 结构化 MDN (Structured MDN)
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, hidden_dim=128, num_components=3, dropout_rate=0.2):
        """
        升级版结构化 MDN：
        - hidden_dim 提升至 128 (为未来扩展到 100+ 特征做准备)
        - 加入 Dropout 防止对某个单一因子过度依赖 (过拟合)
        """
        super(StructuredMDN, self).__init__()
        self.num_components = num_components

        # 通道 A: Macro Network (门控网络 -> 预测机制权重 pi)
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, num_components)
        )

        # 通道 B: Micro Network (专家网络 -> 预测 mu 和 sigma)
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )

        self.mu_head    = nn.Linear(hidden_dim // 2, num_components)
        self.sigma_head = nn.Linear(hidden_dim // 2, num_components)

    def forward(self, x_macro, x_micro):
        pi_logits = self.macro_net(x_macro)
        h_micro   = self.micro_extractor(x_micro)
        mu        = self.mu_head(h_micro)
        sigma     = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6
        return pi_logits, mu, sigma


# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, target, weights):
    """
    weights: shape (batch_size, 1)，距离现在越近权重越大
    """
    log_pi     = torch.log_softmax(pi_logits, dim=-1)
    normal_dist = torch.distributions.Normal(mu, sigma)
    log_normal  = normal_dist.log_prob(target.expand_as(mu))
    log_mix     = log_pi + log_normal
    loss        = -torch.logsumexp(log_mix, dim=-1)
    weighted_loss = loss * weights.squeeze()
    return weighted_loss.mean()


# ==========================================
# [优化 4] 5. 双通道训练循环 —— 新增 ReduceLROnPlateau 调度器
# ==========================================
def train_structured_mdn(model,
                         X_train_macro, X_train_micro, y_train, w_train,
                         X_val_macro,   X_val_micro,   y_val,   w_val,
                         epochs=500, batch_size=8192, lr=1e-3, patience=20):

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # [优化 4] 新增 LR Scheduler：当 val_loss 连续 10 个 epoch 不下降时，lr 乘以 0.5
    # min_lr 防止 lr 降到无意义的极小值
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
    )

    tr_dataset = TensorDataset(X_train_macro, X_train_micro, y_train.unsqueeze(1), w_train.unsqueeze(1))
    va_dataset = TensorDataset(X_val_macro,   X_val_micro,   y_val.unsqueeze(1),   w_val.unsqueeze(1))

    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)

    best_val_loss    = float('inf')
    best_model_state = None
    epochs_no_improve = 0

    train_loss_history = []
    val_loss_history   = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for b_macro, b_micro, b_y, b_w in train_loader:
            optimizer.zero_grad()
            pi_logits, mu, sigma = model(b_macro, b_micro)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)

        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_y, b_w in val_loader:
                pi_logits, mu, sigma = model(b_macro, b_micro)
                loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, b_y, b_w)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
        val_loss_history.append(val_loss)

        # [优化 4] 每个 epoch 结束后，让 scheduler 根据 val_loss 决定是否降 lr
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history


# ==========================================
# 6. 时间衰减权重计算
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    """
    max_date   = df_dates.max()
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    weights      = np.power(decay_factor, months_diff).values
    weights      = weights / weights.mean()  # 归一化，保持 loss 量级稳定
    return weights


# %%
# ==========================================
# 7. 启动模型训练流水线
# ==========================================
if __name__ == "__main__":
    seed_everything(seed=42)
    print("\n>>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...")

    processed_df = processed_df.reset_index(drop=True)
    generator    = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    window_gen   = generator.expanding_window_generator(initial_train_years=15, val_years=1, test_years=1)

    # ==============================================================
    # 全局数据一次性打入 GPU (Global All-in-VRAM)
    # 注意：macro 存的是原始值（未归一化），归一化在循环内做
    # ==============================================================
    print("    -> Pushing the entire dataset into GPU VRAM at once...")
    global_X_mac = torch.FloatTensor(processed_df[macro_tensor_cols].values).to(device)  # 原始值
    global_X_mic = torch.FloatTensor(processed_df[micro_tensor_cols].values).to(device)
    global_Y     = torch.FloatTensor(processed_df['target_ret_final'].values).to(device)
    # ==============================================================

    history_logs       = []
    all_oos_predictions = []
    best_k     = 3
    macro_dim  = len(macro_tensor_cols)
    micro_dim  = len(micro_tensor_cols)
    global_model = StructuredMDN(macro_dim, micro_dim, num_components=best_k).to(device)

    print("\n>>> [3/3] Starting Expanding Window Structured MDN Training...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        start_time = time.time()

        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # ==============================================================
        # [修复 Bug 2] 宏观特征逐窗口归一化
        # 只用当前窗口的训练集来计算 mean/std，再应用到 val 和 test
        # 所有操作都在 GPU 上完成，不产生额外的 CPU-GPU 数据传输
        # ==============================================================
        tr_mac_raw = global_X_mac[idx_tr]
        mac_mean   = tr_mac_raw.mean(dim=0, keepdim=True)   # shape: (1, macro_dim)
        mac_std    = tr_mac_raw.std(dim=0, keepdim=True) + 1e-8

        X_tr_mac = (global_X_mac[idx_tr] - mac_mean) / mac_std
        X_va_mac = (global_X_mac[idx_va] - mac_mean) / mac_std
        X_te_mac = (global_X_mac[idx_te] - mac_mean) / mac_std
        # ==============================================================

        X_tr_mic, Y_tr = global_X_mic[idx_tr], global_Y[idx_tr]
        X_va_mic, Y_va = global_X_mic[idx_va], global_Y[idx_va]
        X_te_mic       = global_X_mic[idx_te]

        w_train_np = calculate_time_decay_weights(train_df['mthcaldt'], half_life_months=36)
        w_val_np   = calculate_time_decay_weights(val_df['mthcaldt'],   half_life_months=12)
        w_train    = torch.FloatTensor(w_train_np).to(device)
        w_val      = torch.FloatTensor(w_val_np).to(device)

        # [设计隐患 3 调整] 第 0 窗口用 1e-3 大步探索；
        # 后续窗口 optimizer 会重置（动量清零），用 3e-4 保留一定探索力度，
        # 比原来的 1e-4 更快找到新数据下的最优解
        current_lr = 1e-3 if window_idx == 0 else 3e-4

        global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
            model=global_model,
            X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, y_train=Y_tr, w_train=w_train,
            X_val_macro=X_va_mac,   X_val_micro=X_va_mic,   y_val=Y_va,   w_val=w_val,
            lr=current_lr
        )

        end_time = time.time()
        duration = end_time - start_time

        history_logs.append({
            'window':        window_idx + 1,
            'train_period':  f"{info['train'][0]} to {info['train'][1]}",
            'test_period':   f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec':  duration,
            'best_val_loss': best_v_loss,
            'epochs_run':    len(tr_hist),
            'train_loss_trace': tr_hist,
            'val_loss_trace':   va_hist
        })

        # 样本外预测
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma = global_model(X_te_mac, X_te_mic)
            pi = torch.softmax(pi_logits, dim=-1)

        # 结果回传 CPU 保存
        # [小改进] 同时保存 pi_logits（原始 logits），方便后续做 calibration / ensemble
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec']      = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['pi_logits_vec'] = [json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()]  # 新增
        preds_df['mu_vec']      = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec']   = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        all_oos_predictions.append(preds_df)

    print("\n>>> Pipeline complete! Aggregating all Out-of-Sample predictions...")
    if len(all_oos_predictions) > 0:
        final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
        metrics_df   = pd.DataFrame(history_logs)
        print(f"Successfully generated {len(final_oos_df)} predictions.")


# %%
# 指定保存路径
kaggle_path = '/kaggle/working'
timestamp   = datetime.now().strftime("%Y%m%d_%H%M")

final_oos_df.to_parquet(os.path.join(kaggle_path, f'final_oos_df_{timestamp}.parquet'), index=False)
metrics_df.to_parquet(os.path.join(kaggle_path,   f'metrics_df_{timestamp}.parquet'),   index=False)

print(f"保存成功！文件位置: {kaggle_path}")